In [3]:
import tensorflow as tf
from kapre import STFT, Magnitude, ApplyFilterbank, MagnitudeToDecibel, STFTTflite, MagnitudeTflite
import numpy as np
from pathlib import Path

import sys
sys.path.append("../")
from genetic_algorithm.utils.convert_to_tflite import convert_to_tflite
from get_ops import get_ops

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2024-01-07 16:17:23.614337: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-01-07 16:17:23.633880: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-01-07 16:17:23.634058: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [4]:
PATH = Path("../tools/test_models_pool/")

if not PATH.exists():
    PATH.mkdir()

In [5]:
def save_as_tflite(model, save_path, quantize, input_shape, print_ops=False):
    if quantize:
        tflite_model = convert_to_tflite(model, np.random.uniform(size=input_shape))
        save_path = str(save_path).replace(".tflite", "_quantized.tflite")
    else:
        tflite_model = convert_to_tflite(model)
        save_path = str(save_path).replace(".tflite", "_non_quantized.tflite")
        
    with open(save_path, 'wb') as f:
        f.write(tflite_model)
        
    if print_ops:
        get_ops(save_path)

## Simple CNN model (input: 28x28x1)

In [6]:
name = "simple_cnn_28x28"
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10, activation='softmax')
])

#model.summary()
model.save(PATH / (name + ".keras"))

save_as_tflite(model, PATH / (name + ".tflite"), quantize=True, input_shape=(1, 28, 28, 1), print_ops=False)
save_as_tflite(model, PATH / (name + ".tflite"), quantize=False, input_shape=(1, 28, 28, 1), print_ops=False)

INFO:tensorflow:Assets written to: /tmp/tmpym7t6xxg/assets


INFO:tensorflow:Assets written to: /tmp/tmpym7t6xxg/assets
/data/du92wufe/Documents/EvoNAS/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
2023-11-20 19:03:02.685007: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2023-11-20 19:03:02.685020: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2023-11-20 19:03:02.685402: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpym7t6xxg
2023-11-20 19:03:02.685994: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2023-11-20 19:03:02.686005: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpym7t6xxg
2023-11-20 19:03:02.688556: I tensorflow/compil

INFO:tensorflow:Assets written to: /tmp/tmpn6bidypq/assets


INFO:tensorflow:Assets written to: /tmp/tmpn6bidypq/assets
2023-11-20 19:03:03.115150: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2023-11-20 19:03:03.115166: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2023-11-20 19:03:03.115270: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpn6bidypq
2023-11-20 19:03:03.115876: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2023-11-20 19:03:03.115887: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpn6bidypq
2023-11-20 19:03:03.117646: I tensorflow/cc/saved_model/loader.cc:228] Restoring SavedModel bundle.
2023-11-20 19:03:03.126397: I tensorflow/cc/saved_model/loader.cc:212] Running initialization op on SavedModel bundle at path: /tmp/tmpn6bidypq
2023-11-20 19:03:03.130827: I tensorflow/cc/saved_model/loader.cc:301] SavedModel

## Simple CNN model (input: 256x256x1)

In [ ]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.summary()

model.save(PATH / "simple_cnn_256x256.keras")

## STFT CNN model (input: 2048x1)

In [125]:
input_shape = (8000, 1)
model = tf.keras.models.Sequential()
model.add(tf.keras.Input(shape=input_shape))

# get number of layers in model
n_layers = len(model.layers)
n_layers

0

In [7]:
NAME = "spoken_languages_1d_and_stft.h5"
model = tf.keras.models.Sequential()

input_shape = (8000, 1)
model.add(tf.keras.Input(shape=input_shape))

# add some 1D convolutions
model.add(tf.keras.layers.DepthwiseConv1D(1, 1, padding='same'))

#model.add(tf.keras.layers.GlobalAveragePooling1D())

model.add(STFTTflite(n_fft=64, hop_length=396,
              input_data_format='channels_last',
              output_data_format='channels_last',
              name='stft'))

model.add(MagnitudeTflite(name='magnitude'))
model.add(tf.keras.layers.DepthwiseConv2D(1, 1, padding='same'))
model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Dense(4, activation='softmax'))

model.summary()
model.save(PATH / NAME, save_format='keras')

save_as_tflite(model, PATH / NAME.replace("h5", "tflite"), quantize=True, input_shape=(1,) + input_shape, print_ops=False)
#save_as_tflite(model, PATH / NAME.replace("keras", "tflite"), quantize=False, input_shape=(1, 6000, 1), print_ops=False)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 depthwise_conv1d_1 (Depthwi  (None, 8000, 1)          2         
 seConv1D)                                                       
                                                                 
 stft (STFTTflite)           (1, 21, 33, 1, 2)         0         
                                                                 
 magnitude (MagnitudeTflite)  (1, 21, 33, 1)           0         
                                                                 
 depthwise_conv2d_1 (Depthwi  (1, 21, 33, 1)           2         
 seConv2D)                                                       
                                                                 
 global_average_pooling2d_1   (1, 1)                   0         
 (GlobalAveragePooling2D)                                        
                                                      

INFO:tensorflow:Assets written to: /tmp/tmpg9rsy_nn/assets


INFO:tensorflow:Assets written to: /tmp/tmpg9rsy_nn/assets
/data/du92wufe/Documents/EvoNAS/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
2024-01-07 16:21:11.038607: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2024-01-07 16:21:11.038627: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2024-01-07 16:21:11.038764: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpg9rsy_nn
2024-01-07 16:21:11.040198: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2024-01-07 16:21:11.040212: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpg9rsy_nn
2024-01-07 16:21:11.045548: I tensorflow/cc/sav


## More networks: TODO: clean-up the code that follows

In [2]:
class FTHelper: 
    
    @staticmethod
    def build_conv_matrix(n_frames): 
        conv_matrix = tf.eye(n_frames)
        conv_matrix = tf.expand_dims(conv_matrix, 0)
        conv_matrix = tf.expand_dims(conv_matrix, -1)
        return conv_matrix

    @staticmethod
    def build_window(frame_length, window_type='hann'): 
        if window_type == 'hann': 
            weights = tf.signal.hann_window(frame_length)
            weights = tf.expand_dims(weights, -1)
            weights = tf.expand_dims(weights, -1)
            weights = tf.expand_dims(weights, 0)
            return tf.sqrt(weights)
        elif window_type == 'ones': 
            return tf.ones([1, frame_length, 1, 1])
        elif window_type == 'uniform': 
            return tf.random.uniform([1, frame_length, 1, 1], minval=0, maxval=1)
        elif window_type == 'random': 
            return tf.random.normal([1, frame_length, 1, 1])
        else: 
            # TODO: Raise error
            return 
    
    @staticmethod
    def build_frequencies(n_freq, freq_type='linspace'): 
        if freq_type == 'linspace': 
            return tf.linspace(0., 3.14, n_freq)
        elif freq_type == 'uniform': 
            return tf.random.uniform([n_freq], minval=0, maxval=3.14)
        else: 
            # TODO: Raise Error
            return 


class STFT(tf.keras.layers.Layer):

    def __init__(
        self, 
        frame_length, 
        frame_step, 
        n_freqs, 
        frequency_trainable=False, 
        window_trainable=False, 
        freq_type='linspace', 
        window_type='hann',
        out_module=True
    ):
        super(STFT, self).__init__()
        self.frame_length = frame_length
        self.frame_step = frame_step
        self.n_freqs = n_freqs
        self.frequency_trainable = frequency_trainable
        self.window_trainable = window_trainable
        self.freq_type = freq_type
        self.window_type = window_type
        self.out_module = out_module
    
    def _call_frame_matrix(self):
        # Convolution2d Transpose
        frame_matrix = tf.squeeze(
            tf.nn.conv2d_transpose(
                input=self.conv_matrix, 
                filters=tf.square(self.window), 
                output_shape=(1, self.n_frames, self.n_y, 1),
                strides=[1, 1, self.frame_step, 1], 
                padding="VALID"
            )
        )
        # Padding
        to_pad = self.sequence_length - self.n_y
        frame_matrix = tf.pad(frame_matrix, tf.constant([[0, 0], [0, to_pad]]), 'CONSTANT')
        return frame_matrix
  
    def build(self, input_shape):
        self.sequence_length = input_shape[-1]
        self.n_frames = (self.sequence_length - self.frame_length) // self.frame_step
        self.n_y = (self.n_frames-1) * self.frame_step + self.frame_length
        self.conv_matrix = FTHelper.build_conv_matrix(self.n_frames)
        self.times = tf.range(self.sequence_length, dtype=tf.float32)
        self.window = tf.Variable(
            name='window', 
            initial_value=FTHelper.build_window(self.frame_length, self.window_type),
            trainable=self.window_trainable
        )
        self.frequencies = tf.Variable(
            name='frequencies',
            initial_value=FTHelper.build_frequencies(self.n_freqs, self.freq_type), 
            trainable=self.frequency_trainable
        )

    def call(self, inputs):
        frame_matrix = self._call_frame_matrix()
        frames = tf.multiply(
            tf.expand_dims(tf.transpose(frame_matrix), 0), 
            tf.expand_dims(inputs, -1), 
        )
        frames = tf.transpose(frames, perm=[0, 2, 1])
        # DFT Computations
        operands = tf.tensordot(self.times, self.frequencies, axes=0)
        w_real = tf.math.cos(-operands)
        w_im = tf.math.sin(-operands)
        y_real = tf.tensordot(frames, w_real, axes=1)
        y_im = tf.tensordot(frames, w_im, axes=1)
        if self.out_module: 
            y_mod = tf.add(tf.square(y_real), tf.square(y_im))
            return tf.expand_dims(tf.math.log(y_mod + 1) / 2, -1)
        else: 
            return tf.stack([y_real, y_im], axis=-1) / self.sequence_length



In [3]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=6_000),
    STFT(frame_length=512, frame_step=128, n_freqs=40, out_module=False)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

2023-10-30 13:43:19.170897: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.229241: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.229376: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-10-30 13:43:19.230224: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the approp

In [4]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 stft (STFT)                 (None, 42, 40, 2)         552       
                                                                 
Total params: 552
Trainable params: 0
Non-trainable params: 552
_________________________________________________________________


In [37]:
class ExpandShapeLayer(tf.keras.layers.Layer):
    def build(self, input_shape):
        pass
    
    def call(self, inputs):
        return tf.expand_dims(inputs, axis=-1)

In [39]:
model = tf.keras.Sequential()

#model.add(tf.keras.Input(shape=6000))
model.add(tf.keras.Input(shape=(6000, 1)))

model.add(STFTTflite(n_fft=400, hop_length=160,
              input_data_format='channels_last',
              output_data_format='channels_last',
              input_shape=(6000, 1),
              name='stft'))

#model.add(TF_STFT_Layer(name='stft'))
model.add(STFT(frame_length=512, frame_step=128, n_freqs=40, out_module=False))

print(model.output_shape)
# reshape
#model.add(tf.keras.layers.Reshape((36, 257, 1)))

model.add(MagnitudeTflite(name='magnitude'))

# add dimension at the end for channels
model.add(ExpandShapeLayer())

print(model.output_shape)

model.add(tf.keras.layers.Conv2D(32, 3))
model.add(tf.keras.layers.ReLU())
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Conv2D(16, 3))
model.add(tf.keras.layers.ReLU())
model.add(tf.keras.layers.BatchNormalization())

model.add(tf.keras.layers.GlobalAveragePooling2D())
model.add(tf.keras.layers.Dense(12, activation='relu'))

(None, 42, 40, 2)
(None, 42, 40, 1)


In [40]:
model.summary()

Model: "sequential_21"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 stft_17 (STFT)              (None, 42, 40, 2)         552       
                                                                 
 magnitude (MagnitudeTflite)  (None, 42, 40)           0         
                                                                 
 expand_shape_layer (ExpandS  (None, 42, 40, 1)        0         
 hapeLayer)                                                      
                                                                 
 conv2d_15 (Conv2D)          (None, 40, 38, 32)        320       
                                                                 
 re_lu_8 (ReLU)              (None, 40, 38, 32)        0         
                                                                 
 batch_normalization_8 (Batc  (None, 40, 38, 32)       128       
 hNormalization)                                     

In [41]:
tflite_model = convert_to_tflite(model, np.random.uniform(size=(1, 6_000)))
#tflite_model = convert_to_tflite(model)

INFO:tensorflow:Assets written to: /tmp/tmpkihf4ifx/assets


INFO:tensorflow:Assets written to: /tmp/tmpkihf4ifx/assets
/data/du92wufe/Documents/EvoNAS/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:766: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn("Statistics for quantized inputs were expected, but not "
2023-10-30 15:38:10.318566: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:362] Ignored output_format.
2023-10-30 15:38:10.318583: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:365] Ignored drop_control_dependency.
2023-10-30 15:38:10.318678: I tensorflow/cc/saved_model/reader.cc:43] Reading SavedModel from: /tmp/tmpkihf4ifx
2023-10-30 15:38:10.320248: I tensorflow/cc/saved_model/reader.cc:81] Reading meta graph with tags { serve }
2023-10-30 15:38:10.320261: I tensorflow/cc/saved_model/reader.cc:122] Reading SavedModel debug info (if present) from: /tmp/tmpkihf4ifx
2023-10-30 15:38:10.325128: I tensorflow/cc/sav

In [42]:
with open('test_model_custom_stft.tflite', 'wb') as f:
    f.write(tflite_model)

In [52]:
get_ops('test_model.tflite')

['SHAPE (0)',
 'SPLIT_V (1)',
 'RESHAPE (2)',
 'FLOOR_DIV (3)',
 'PACK (4)',
 'MUL (5)',
 'RESHAPE (2)',
 'CONCATENATION (6)',
 'CONCATENATION (6)',
 'STRIDED_SLICE (7)',
 'RESHAPE (2)',
 'SUB (8)',
 'FLOOR_DIV (3)',
 'ADD (9)',
 'MAXIMUM (10)',
 'PACK (4)',
 'PACK (4)',
 'CONCATENATION (6)',
 'RANGE (11)',
 'MUL (5)',
 'RESHAPE (2)',
 'ADD (9)',
 'GATHER (12)',
 'RESHAPE (2)',
 'MUL (5)',
 'PAD (13)',
 'EXPAND_DIMS (14)',
 'DEQUANTIZE (15)',
 'RFFT2D (16)',
 'SQUEEZE (17)',
 'SHAPE (0)',
 'STRIDED_SLICE (7)',
 'PACK (4)',
 'RESHAPE (2)',
 'COMPLEX_ABS (18)',
 'QUANTIZE (19)',
 'CONV_2D (20)',
 'MUL (5)',
 'ADD (9)',
 'MAX_POOL_2D (21)',
 'CONV_2D (20)',
 'MUL (5)',
 'ADD (9)',
 'MEAN (22)',
 'FULLY_CONNECTED (23)']

In [44]:
get_ops('test_model_custom_stft.tflite')

['EXPAND_DIMS (0)',
 'MUL (1)',
 'TRANSPOSE (2)',
 'SHAPE (3)',
 'GATHER (4)',
 'REDUCE_PROD (5)',
 'CONCATENATION (6)',
 'GATHER (4)',
 'REDUCE_PROD (5)',
 'PACK (7)',
 'RESHAPE (8)',
 'FULLY_CONNECTED (9)',
 'RESHAPE (8)',
 'FULLY_CONNECTED (9)',
 'RESHAPE (8)',
 'PACK (7)',
 'MUL (1)',
 'MUL (1)',
 'SUM (10)',
 'DEQUANTIZE (11)',
 'SQRT (12)',
 'QUANTIZE (13)',
 'SQUEEZE (14)',
 'EXPAND_DIMS (0)',
 'CONV_2D (15)',
 'MUL (1)',
 'ADD (16)',
 'MAX_POOL_2D (17)',
 'CONV_2D (15)',
 'MUL (1)',
 'ADD (16)',
 'MEAN (18)',
 'FULLY_CONNECTED (9)']